In [1]:
import os 
os.chdir('../../../../')
os.environ["DPM_TQDM"] = "False"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

!nvidia-smi

Mon Aug 11 22:04:56 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:19:00.0 Off |                  Off |
|  0%   51C    P8             41W /  450W |      11MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# -*- coding: utf-8 -*-
import os
import math
import numpy as np
from easydict import EasyDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

# ===============================
# Config
# ===============================
config = EasyDict()
config.backbone      = 'DiT'
config.train_pt_dir  = 'samplings/dit/train_4.0/dit_train_4.0_1'
config.valid_pt_dir  = 'samplings/dit/eval1000_4.0/dit_eval1000_4.0_0'
config.batch_size    = 10
config.CFG           = 4.0
config.epochs        = 10
config.val_every     = 100
config.log_dir       = "logs/CFG4.0/0811-11:OneCycleLR"

# LR & Scheduler
config.base_lr       = 2e-3
config.total_steps   = 1000        # 전체 학습 스텝
config.warmup_steps  = 50          # 워ーム업 스텝
config.min_lr_ratio  = 0.10        # 코사인 최저 비율 (= base_lr * 0.10)

os.makedirs(config.log_dir, exist_ok=True)

# ===============================
# Model (frozen)
# ===============================
from backbones.dit import DiT

if config.backbone == 'DiT':
    model = DiT(trainable=True)  # 내부 구현에 맞춰 유지
    model.set_freeze()
device = model.device
print(model)

# ===============================
# Dataset / Dataloader
# ===============================
from datasets.pt_dataset import PtDataset

train_dataset = PtDataset(config.train_pt_dir)
valid_dataset = PtDataset(config.valid_pt_dir)
print('len(train_dataset) :', len(train_dataset), 'len(valid_dataset) :', len(valid_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

valid_loader = DataLoader(valid_dataset, batch_size=config.batch_size, shuffle=False)
print('dataloaders ready')

# ===============================
# Solver / Optimizer / Scheduler
# ===============================
from solvers.dual.dynamic.gdual_solver_log import GDual_Solver
from solvers.transforms.loglinear_transform import LogLinearTransform
from solvers.param_extractors.gap_extractor import GAP_Extractor
from torch.optim.lr_scheduler import OneCycleLR

noise_schedule = model.get_noise_schedule()
extractor = GAP_Extractor(hidden_dim=128, input_shape=(4, 32, 32))
solver = GDual_Solver(
    noise_schedule,
    steps=5,
    transform=LogLinearTransform,
    param_extractor=extractor,
    exact_first=False,
    skip_type="time_uniform",
).to(device)

# OneCycle에서는 optimizer의 초기 lr을 max_lr로 두는 게 헷갈림이 적다
optimizer = torch.optim.AdamW(solver.parameters(), lr=config.base_lr)

# warmup 비율 = warmup_steps / total_steps  (예: 50/1000 = 0.05)
pct_start = config.warmup_steps / config.total_steps

scheduler = OneCycleLR(
    optimizer,
    max_lr=config.base_lr,             # 피크 LR
    total_steps=config.total_steps,    # 총 스텝(= 네가 정한 1000)
    pct_start=pct_start,               # 워ーム업 구간 비율
    anneal_strategy="cos",             # 코사인 다운스윙
    div_factor=10.0,                   # 시작 lr = max_lr / 10
    final_div_factor=10.0,             # 마지막 lr = 시작 lr / 10
    cycle_momentum=False,              # AdamW는 False 권장
)
print('solver/optimizer/OneCycleLR ready')

# ===============================
# Utils
# ===============================
def save_checkpoint(global_step, save_dir, solver, valid_loss):
    ckpt = {
        "global_step": int(global_step),
        "solver_state_dict": solver.state_dict(),
        "valid_loss": float(valid_loss),
        "config": dict(config),
    }
    os.makedirs(save_dir, exist_ok=True)
    step_path = os.path.join(save_dir, f"step_{global_step:08d}.pt")
    torch.save(ckpt, step_path)
    return step_path

@torch.no_grad()
def get_valid_loss(device, solver):
    solver.eval()
    losses = []
    pbar = tqdm(valid_loader, leave=False)
    for batch in pbar:
        noises = batch['noise'].to(device, non_blocking=True)
        conds  = batch['cond']
        targets= batch['sample'].to(device, non_blocking=True)
        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)
        pred = solver.sample(noises, model_fn)
        loss = F.mse_loss(pred, targets)
        losses.append(loss.item())
        pbar.set_postfix({'val_loss': loss.item()})
    return float(np.mean(losses))

def do_train_loop(device, epoch, writer, solver, optimizer, scheduler, global_step_start=0):
    solver.train()
    pbar = tqdm(train_loader)
    losses = []
    global_step = global_step_start

    for step, batch in enumerate(pbar):
        if global_step >= config.total_steps:
            break

        if global_step % config.val_every == 0:
            val = get_valid_loss(device, solver)
            print(f'step : {global_step} valid_loss : {val:.6f}')
            writer.add_scalar("valid/loss", val, global_step)
            save_checkpoint(global_step, config.log_dir, solver, val)

        optimizer.zero_grad(set_to_none=True)
        noises = batch['noise'].to(device, non_blocking=True)
        conds  = batch['cond']
        targets= batch['sample'].to(device, non_blocking=True)

        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            pred = solver.sample(noises, model_fn)
            loss = F.mse_loss(pred, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(solver.parameters(), 1.0)
        optimizer.step()
        scheduler.step()  # <- after optimizer.step()

        # logging
        lr_now = optimizer.param_groups[0]["lr"]
        writer.add_scalar("train/lr", lr_now, global_step)
        writer.add_scalar("train/loss", loss.item(), global_step)

        losses.append(loss.item())
        pbar.set_postfix({'loss': loss.item(), 'lr': lr_now})
        global_step += 1

    return float(np.mean(losses)) if losses else 0.0, global_step


/home/scpark/miniconda3/envs/rbf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]An error occurred while trying to fetch /home/scpark/.cache/huggingface/hub/models--facebook--DiT-XL-2-256/snapshots/eab87f77abd5aef071a632f08807fbaab0b704d0/transformer: Error no file named diffusion_pytorch_model.safetensors found in directory /home/scpark/.cache/huggingface/hub/models--facebook--DiT-XL-2-256/snapshots/eab87f77abd5aef071a632f08807fbaab0b704d0/transformer.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  67%|██████▋   | 2/3 [00:00<00:00,  4.80it/s]An error occurred while trying to fetch /home/scpark/.cache/huggingface/hub/models--facebook--DiT-XL-2-256/snapshots/eab8

len(train_dataset) : 10000 len(valid_dataset) : 1000
dataloaders ready
solver/optimizer/OneCycleLR ready


In [3]:
# ===============================
# Train
# ===============================
def main():
    writer = SummaryWriter(config.log_dir)
    print('tensorboard:', config.log_dir)

    global_step = 0
    for epoch in range(config.epochs):
        if global_step >= config.total_steps:
            break
        mean_loss, global_step = do_train_loop(
            device, epoch, writer, solver, optimizer, scheduler, global_step_start=global_step
        )
        print(f'[epoch {epoch}] mean_train_loss={mean_loss:.6f}, global_step={global_step}')

    # 마지막 검증 & 체크포인트
    val = get_valid_loss(device, solver)
    save_checkpoint(global_step, config.log_dir, solver, val)
    writer.add_scalar("valid/loss_final", val, global_step)
    writer.close()
    print('done')

if __name__ == "__main__":
    main()


tensorboard: logs/CFG4.0/0811-11:OneCycleLR


  0%|          | 0/1000 [00:00<?, ?it/s]

step : 0 valid_loss : 1.997355


 10%|█         | 100/1000 [02:00<13:56,  1.08it/s, loss=0.291, lr=0.00199] 

step : 100 valid_loss : 0.317641


...

[NaNCheck][step -1] timesteps: NaN=5, Inf=0, finite[min=1.000e+00, max=1.000e+00, mean=1.000e+00], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] alphas: NaN=5, Inf=0, finite[min=6.330e-03, max=6.330e-03, mean=6.330e-03], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] sigmas: NaN=5, Inf=0, finite[min=1.000e+00, max=1.000e+00, mean=1.000e+00], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] log_alpha: NaN=5, Inf=0, finite[min=-5.062e+00, max=-5.062e+00, mean=-5.062e+00], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] log_sigma: NaN=5, Inf=0, finite[min=-2.003e-05, max=-2.003e-05, mean=-2.003e-05], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] log_alpha_ratio: NaN=5, Inf=0, finite[min=nan, max=nan, mean=nan], shape=(5,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] log_sigma_ratio: NaN=5, Inf=0, finite[min=nan, max=nan, mean=nan], shape=(5,), dtype=torch.float32, device=cuda:0
[N

 26%|██▋       | 263/1000 [05:48<16:51,  1.37s/it, loss=nan, lr=0.00176]

[NaNCheck][step -1] timesteps: NaN=5, Inf=0, finite[min=1.000e+00, max=1.000e+00, mean=1.000e+00], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] alphas: NaN=5, Inf=0, finite[min=6.330e-03, max=6.330e-03, mean=6.330e-03], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] sigmas: NaN=5, Inf=0, finite[min=1.000e+00, max=1.000e+00, mean=1.000e+00], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] log_alpha: NaN=5, Inf=0, finite[min=-5.062e+00, max=-5.062e+00, mean=-5.062e+00], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] log_sigma: NaN=5, Inf=0, finite[min=-2.003e-05, max=-2.003e-05, mean=-2.003e-05], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] log_alpha_ratio: NaN=5, Inf=0, finite[min=nan, max=nan, mean=nan], shape=(5,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] log_sigma_ratio: NaN=5, Inf=0, finite[min=nan, max=nan, mean=nan], shape=(5,), dtype=torch.float32, device=cuda:0
[N

 26%|██▋       | 263/1000 [05:49<16:20,  1.33s/it, loss=nan, lr=0.00176]


KeyboardInterrupt: 